# Tutorial 2: Simulation of FOV-Induced Incomplete Registration in Human Lymph Node

This tutorial uses paired RNA and ADT profiles from human lymph node, available through the [SpaMosaic dataset on Zenodo](https://zenodo.org/records/18946723). In this source-to-target setting, RNA is retained as the complete source modality and a contiguous ADT region is masked to simulate FOV-induced incomplete registration.


In [ ]:
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (compute_similarity_prior, plot_per_protein_correlations, plot_prism_imputation_spatial,
                   preprocess_omics, prism_eval_and_save, run_clustering_eval_plot, select_best_device,
                   set_prism_plot_style, set_seed, simulate_missing_sliding)
set_prism_plot_style()

In [ ]:
# Load data and set up paths
DEVICE = select_best_device()
RANDOM_SEED = 2024
set_seed(RANDOM_SEED)
Slice_ID = "S1"

DATASET_DIR = Path("Datasets") / "human lymphoid organs" / "lymph" / Slice_ID
SOURCE_H5AD = DATASET_DIR / "adata_RNA.h5ad"    
TARGET_H5AD = DATASET_DIR / "adata_ADT.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial2_1_human_lymph_node"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_PATH = RESULTS_DIR / f"{Slice_ID}_lymph_AOT.npz"
RUN_PREFIX = f"{Slice_ID}_lymph"

adata_source = sc.read_h5ad(SOURCE_H5AD)
adata_target = sc.read_h5ad(TARGET_H5AD)
adata_source.var_names_make_unique()
adata_target.var_names_make_unique()

In [ ]:
# Simulating FOV-induced incompleteness
missing_indices = simulate_missing_sliding(adata_target, spatial_key="spatial", direction="H", missing_width=0.50, 
                                           step_ratio=0.10, step_id=0, label_key="missing", lock_at_end=True,
                                           point_size=4, plot=True, figsize=(4, 4))

print(f"ADT missing cells: {len(missing_indices)}/{adata_target.n_obs}")

In [ ]:
# Preprocessing source and target modalities
adata_source, _ = preprocess_omics(adata_source, modality="RNA", missing_key="missing", min_cells=10, 
                                   hvgs=3000, data_role="source", compute_pca=False, save_raw_eval=False)

adata_target, _ = preprocess_omics(adata_target, modality="ADT", missing_key="missing", data_role="target",
                                   compute_pca=False, save_log_layer=False)

print("RNA shape after preprocessing:", adata_source.shape)
print("ADT shape after preprocessing:", adata_target.shape)

In [ ]:
# Constructing the RNA similarity prior
distance_matrix, prior_metrics = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH, device=DEVICE,
                                                          covet_k_spatial=6, covet_gene_num=64, spatial_key="spatial", 
                                                          missing_key="missing", store_neighbor_index=True)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_target)

In [ ]:
# Train PRISM
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix, 
                                                       k_top=5, n_epochs=1000, lr=1e-3, 
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX,
                                                       device=DEVICE, patience=20, min_epochs=50,
                                                       center_drop_rate=0.1, noise=0.0,
                                                       load_model_path=False, interaction_pca=True)

### Task 1: Spatial-domain identification

In [ ]:
# Task 1: Spatial-domain identification
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb", label_key="final_annot",
                                                           cluster_key="PRISM_mclust", n_clusters=10, s=20, use_pca=True,
                                                           align_labels=True, aligned_key="PRISM_mclust_domain",
                                                           dataset_name="lymph")

### Task 2: ADT imputation

In [ ]:
# Evaluate ADT imputation in the simulated missing FOV
imputation_results = prism_eval_and_save(truth_adata=adata_target_out, adata=adata_target_out, 
                                         save_path=str(RESULTS_DIR), first_name=RUN_PREFIX,
                                         missing_indices=missing_indices,
                                         save_files=True)

# Visualize per-protein PCC and SPCC across missing cells
per_protein_figures = plot_per_protein_correlations(imputation_results, feature_names=adata_target.var_names)

In [ ]:
#  Visualize Representative ADT feature
FEATURE_TO_PLOT = "CD3E"

fig, axs = plot_prism_imputation_spatial(imputation_results=imputation_results, split1_indices=missing_indices,
                                         feature=FEATURE_TO_PLOT, show_missing_only=False, highlight_missing=False,
                                         figsize=(8, 3))

feature_idx = adata_target.var_names.get_loc(FEATURE_TO_PLOT)
feature_metrics = {metric: round(float(imputation_results["raw"]["per_protein"][metric][feature_idx]), 4)
                   for metric in ("PCC", "SPCC", "MSE")}
print(f"Representative ADT protein {FEATURE_TO_PLOT}: {feature_metrics}")